# Used-car price prediction: exploratory data analysis

This notebook focuses only on exploratory analysis of the cleaned Car Price Assignment dataset (Data 3). It does not train models or create train/test splits. The cleaning step is reproducible in `src/preprocess_data3.py`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

In [ ]:
def first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists():
            return candidate
    raise FileNotFoundError("None of these dataset paths exists:\n" + "\n".join(map(str, paths)))


df_3_path = first_existing([
    "data/processed/data3_cleaned.csv",
    "../data/processed/data3_cleaned.csv",
    "/kaggle/input/datasets/mazenbassem/resources/data/processed/data3_cleaned.csv",
    "/kaggle/input/car-pricev3/data3_cleaned.csv",
])
df_3 = pd.read_csv(df_3_path)
print(f"Data 3 loaded from: {df_3_path}")
print(f"Rows: {len(df_3):,} | Columns: {df_3.shape[1]}")

## 1. Dataset overview and quality checks

In [ ]:
display(df_3.head())
print("Data types:")
display(df_3.dtypes.rename("dtype").to_frame())
print("Duplicate rows:", int(df_3.duplicated().sum()))
quality = pd.DataFrame({
    "dtype": df_3.dtypes.astype(str),
    "missing": df_3.isna().sum(),
    "missing_pct": (df_3.isna().mean() * 100).round(2),
    "unique_values": df_3.nunique(),
})
display(quality)
print("Summary statistics:")
display(df_3.describe(include="all").T)

## 2. Univariate distributions

The plots below show the shape, spread, and possible skew of the numerical variables.

In [ ]:
numeric_columns = df_3.select_dtypes(include=np.number).columns.tolist()
numeric_columns.remove("car_id")

fig, axes = plt.subplots(4, 4, figsize=(18, 16))
for ax, column in zip(axes.flat, numeric_columns):
    sns.histplot(df_3[column], kde=True, bins=20, ax=ax)
    ax.set_title(column.replace("_", " ").title())
for ax in axes.flat[len(numeric_columns):]:
    ax.remove()
fig.suptitle("Numerical feature distributions", y=1.02, fontsize=16)
plt.tight_layout()

In [ ]:
categorical_columns = [
    "brand", "fuel_type", "aspiration", "doors", "body_style", "drive_wheel",
    "engine_location", "engine_type", "cylinders", "fuel_system",
]

fig, axes = plt.subplots(5, 2, figsize=(16, 22))
for ax, column in zip(axes.flat, categorical_columns):
    counts = df_3[column].value_counts().sort_values()
    sns.barplot(x=counts.values, y=counts.index, ax=ax, color="steelblue")
    ax.set_title(f"{column.replace('_', ' ').title()} frequency")
    ax.set_xlabel("Number of cars")
    ax.set_ylabel("")
plt.tight_layout()

## 3. Price analysis

Price is the target variable for the later modeling stage, but here it is examined descriptively only.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df_3["price"], kde=True, bins=25, ax=axes[0], color="darkgreen")
axes[0].set_title("Price distribution")
sns.boxplot(x=df_3["price"], ax=axes[1], color="lightgreen")
axes[1].set_title("Price spread and outliers")
sns.histplot(np.log1p(df_3["price"]), kde=True, bins=25, ax=axes[2], color="teal")
axes[2].set_title("Log-transformed price distribution")
plt.tight_layout()

display(df_3["price"].describe().to_frame("price"))

In [ ]:
grouped_price = (
    df_3.groupby("body_style", as_index=False)
    .agg(car_count=("price", "size"), median_price=("price", "median"),
         mean_price=("price", "mean"), min_price=("price", "min"), max_price=("price", "max"))
    .sort_values("median_price", ascending=False)
)
display(grouped_price)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=df_3, x="body_style", y="price", ax=axes[0])
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_title("Price by body style")
sns.boxplot(data=df_3, x="fuel_type", y="price", ax=axes[1])
axes[1].set_title("Price by fuel type")
plt.tight_layout()

In [ ]:
top_brands = (
    df_3.groupby("brand", as_index=False)
    .agg(car_count=("price", "size"), median_price=("price", "median"), mean_price=("price", "mean"))
    .query("car_count >= 3")
    .sort_values("median_price", ascending=False)
)
display(top_brands)

plt.figure(figsize=(14, 7))
sns.boxplot(data=df_3, x="brand", y="price", order=top_brands["brand"])
plt.xticks(rotation=45, ha="right")
plt.title("Price distribution by brand (brands with at least 3 cars)")
plt.tight_layout()

## 4. Relationships between numerical variables

In [ ]:
relationship_columns = [
    "price", "horsepower", "engine_size", "curb_weight", "car_width",
    "wheelbase", "city_mpg", "highway_mpg", "compression_ratio",
]
correlations = df_3[relationship_columns].corr()
plt.figure(figsize=(11, 8))
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.tight_layout()

display(correlations["price"].sort_values(ascending=False).to_frame("correlation_with_price"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=df_3, x="horsepower", y="price", hue="fuel_type", ax=axes[0])
axes[0].set_title("Price vs horsepower")
sns.scatterplot(data=df_3, x="engine_size", y="price", hue="aspiration", ax=axes[1])
axes[1].set_title("Price vs engine size")
sns.scatterplot(data=df_3, x="curb_weight", y="price", hue="body_style", ax=axes[2])
axes[2].set_title("Price vs curb weight")
plt.tight_layout()

## 5. Segment comparisons and observations

These summaries support hypotheses for later modeling; they are descriptive and do not establish causation.

In [ ]:
segment_summary = (
    df_3.groupby(["drive_wheel", "aspiration"], as_index=False)
    .agg(car_count=("price", "size"), median_price=("price", "median"),
         median_horsepower=("horsepower", "median"), median_mpg=("highway_mpg", "median"))
    .sort_values("median_price", ascending=False)
)
display(segment_summary)

print("Highest-priced observations:")
display(df_3.nlargest(10, "price")[["brand", "model", "body_style", "horsepower", "engine_size", "price"]])
print("Lowest-priced observations:")
display(df_3.nsmallest(10, "price")[["brand", "model", "body_style", "horsepower", "engine_size", "price"]])

## EDA takeaways

- Review the displayed missing-value table before feature engineering.
- Compare medians as well as means because vehicle prices are typically skewed.
- Interpret correlations as associations; brand, body style, and engine characteristics may interact.
- The next notebook stage can use these findings to define features and modeling questions, but no model is trained here.